In [ ]:
import pandas as pd

# Load data using relative paths
brands = pd.read_csv('../data/raw/brands.csv')
orderlines = pd.read_csv('../data/raw/orderlines.csv')
orders = pd.read_csv('../data/raw/orders.csv')
products = pd.read_csv('../data/raw/products.csv')

# Create working copies
brands_df = brands.copy()
orderlines_df = orderlines.copy()
orders_df = orders.copy()
products_df = products.copy()

In [ ]:
products_df_copy = products_df.copy()

In [ ]:
products_df_copy.shape

In [ ]:
products_df_copy.columns

In [ ]:
nrows = products_df_copy.shape[0]
ncols = products_df_copy.shape[1]
print("The number of rows is", nrows)
print("The number of columns is", ncols)

In [ ]:
products_df_copy.size

In [ ]:
products_df_copy.shape[0] * products_df_copy.shape[1] == products_df_copy.size

In [ ]:
products_df_copy.ndim

In [ ]:
products_df_copy.head(20)

In [ ]:
products_df_copy.tail(20)

In [ ]:
products_df_copy.info()

In [ ]:
products_df_copy['price'].describe()

In [ ]:
mask = products_df_copy['type'].duplicated(keep=False)
products_df_copy.loc[mask,:]

In [ ]:
products_df_copy.isna().sum()

In [ ]:
desc_mask = products_df_copy['desc'].isna()
products_df_copy.loc[desc_mask, 'desc'] = products_df_copy.loc[desc_mask, 'name']
products_df_copy['desc'].isna().sum()

In [ ]:
mask = products_df_copy['type'].isna()
products_df_copy.loc[mask,:]

In [ ]:
products_df_copy.info()
price_mask = products_df_copy["price"].str.count(r"\.") > 1
promo_price_mask = products_df_copy["promo_price"].str.count(r"\.") > 1
print(len(products_df_copy))
print(len(products_df_copy.loc[price_mask, :]))
print(len(products_df_copy.loc[promo_price_mask, :]))
print(
    f"Percentage of price_mask: {round(len(products_df_copy.loc[price_mask,:]) / len(products_df_copy)*100,3)}\nPercentage of promo_price_mask: {round(len(products_df_copy.loc[promo_price_mask,:]) / len(products_df_copy)*100,3)}\n"
)

In [ ]:
products_df_copy.loc[price_mask,:].sample(20)

In [ ]:
products_df_copy.loc[promo_price_mask,:].sample(20)

In [ ]:
products_df_copy.loc[price_mask | promo_price_mask,:].sample(30)

In [ ]:
products_df_copy.info()
mask_comma = products_df_copy["type"].str.contains(r"\,")
mask_plus = products_df_copy["type"].str.contains(r"\+")
type_mask = mask_comma & mask_plus
print(len(products_df_copy))
print(len(products_df_copy.loc[type_mask, :]))
print(
    f"Percentage of type: {round(len(products_df_copy.loc[type_mask,:]) / len(products_df_copy)*100,3)}"
)

In [ ]:
print(f'there are {len(products_df_copy.loc[(price_mask | promo_price_mask) & type_mask, :])} rows with scientific types and price problems')
print(f'there are {len(products_df_copy.loc[(price_mask | promo_price_mask) & ~(type_mask), :])} rows without scientific types and price problems')

In [ ]:
orderlines_df.head()
merged_df = products_df_copy.merge(orderlines_df, on='sku')
merged_df.sample(20)

In [ ]:
merged_df.loc[merged_df["promo_price"].str.count(r"\.") > 1,:].sample(20)

In [ ]:
merged_df.loc[merged_df["unit_price"].str.count(r"\.") == 1,:].sample(20)

In [ ]:
merged_df['new_unit_price'] = merged_df['unit_price']
mask = merged_df["unit_price"].str.count(r"\.") > 1
merged_df.loc[mask, "new_unit_price"] = merged_df.loc[mask, "unit_price"].str.replace(pat=r'\.',repl='',n=1,regex=True)
merged_df["new_unit_price"] = pd.to_numeric(merged_df["new_unit_price"])
merged_df = merged_df.loc[:,['sku', 'name', 'desc', 'price', 'promo_price', 'new_unit_price']]


In [ ]:
price_exploration_df = merged_df.copy()
price_mask = price_exploration_df["price"].str.count(r"\.") > 1
price_exploration_df['price_replace'] = price_exploration_df.loc[price_mask, "price"].str.replace(pat=r'\.',repl='',regex=True)
price_exploration_df

In [ ]:
# Promo Price Pattern Analysis
# Note: new_unit_price was already created in the previous cell

# Step 1: Verify new_unit_price exists
print("\n1. Checking new_unit_price column...")
print(f"   new_unit_price column exists: {'new_unit_price' in merged_df.columns}")
print(f"   Sample values: {merged_df['new_unit_price'].head(3).tolist()}")

# Step 2: Identify problematic rows
print("\n2. Identifying problematic rows...")
price_mask = merged_df['price'].str.count(r'\.') > 1
promo_price_mask = merged_df['promo_price'].str.count(r'\.') > 1
print(f"   - Price problems: {price_mask.sum()} ({price_mask.sum()/len(merged_df)*100:.1f}%)")
print(f"   - Promo price problems: {promo_price_mask.sum()} ({promo_price_mask.sum()/len(merged_df)*100:.1f}%)")

# Step 3: Analyze promo_price patterns
# Test 1: Remove first dot
problematic_promo = merged_df[promo_price_mask].copy()
problematic_promo['test_remove_first_dot'] = problematic_promo['promo_price'].str.replace(r'\.', '', n=1, regex=True)
problematic_promo['test_numeric'] = pd.to_numeric(problematic_promo['test_remove_first_dot'], errors='coerce')
problematic_promo['ratio_to_unit'] = problematic_promo['test_numeric'] / problematic_promo['new_unit_price']

# How many are "reasonable"? (Promo price should be 0.5-1.5x unit_price)
reasonable = (problematic_promo['ratio_to_unit'] >= 0.5) & (problematic_promo['ratio_to_unit'] <= 1.5)
print(f"\n   TEST 1: Remove first dot")
print(f"   -> {reasonable.sum()} / {len(problematic_promo)} rows have reasonable ratio (0.5-1.5x)")
print(f"   -> Success rate: {reasonable.sum()/len(problematic_promo)*100:.1f}%")

# Show examples
print("\n   Examples:")
print(problematic_promo[['promo_price', 'new_unit_price', 'test_remove_first_dot', 'ratio_to_unit']].head(10))

# Test 2: Remove ALL dots, then divide by 100
problematic_promo['test_remove_all_dots'] = problematic_promo['promo_price'].str.replace(r'\.', '', regex=True)
problematic_promo['test_numeric_2'] = pd.to_numeric(problematic_promo['test_remove_all_dots'], errors='coerce') / 100
problematic_promo['ratio_to_unit_2'] = problematic_promo['test_numeric_2'] / problematic_promo['new_unit_price']

reasonable_2 = (problematic_promo['ratio_to_unit_2'] >= 0.5) & (problematic_promo['ratio_to_unit_2'] <= 1.5)
print(f"\n   TEST 2: Remove all dots, divide by 100")
print(f"   -> {reasonable_2.sum()} / {len(problematic_promo)} rows have reasonable ratio")
print(f"   -> Success rate: {reasonable_2.sum()/len(problematic_promo)*100:.1f}%")

# Step 4: Specific Pattern Recognition
print("\n4. Specific pattern analysis...")
print("-"*80)

# Pattern 1: X.XXX.XXX (e.g., 7.640.001)
pattern_1 = problematic_promo[problematic_promo['promo_price'].str.match(r'^\d\.\d{3}\.\d{3}$', na=False)].copy()
print(f"\n   PATTERN: X.XXX.XXX (like 7.640.001)")
print(f"   -> Found {len(pattern_1)} rows")
if len(pattern_1) > 0:
    pattern_1['fixed'] = pd.to_numeric(pattern_1['promo_price'].str.replace(r'\.', '', n=1, regex=True))
    pattern_1['ratio'] = pattern_1['fixed'] / pattern_1['new_unit_price']
    reasonable = (pattern_1['ratio'] >= 0.5) & (pattern_1['ratio'] <= 1.5)
    print(f"   -> {reasonable.sum()} are reasonable ({reasonable.sum()/len(pattern_1)*100:.1f}%)")
    print("\n   Sample:")
    print(pattern_1[['promo_price', 'new_unit_price', 'fixed', 'ratio']].head(5))

# Pattern 2: XX.XXX.XXX (e.g., 11.590.009)
pattern_2 = problematic_promo[problematic_promo['promo_price'].str.match(r'^\d{2}\.\d{3}\.\d{3}$', na=False)].copy()
print(f"\n   PATTERN: XX.XXX.XXX (like 11.590.009)")
print(f"   -> Found {len(pattern_2)} rows")
if len(pattern_2) > 0:
    pattern_2['fixed'] = pd.to_numeric(pattern_2['promo_price'].str.replace(r'\.', '', n=1, regex=True))
    pattern_2['ratio'] = pattern_2['fixed'] / pattern_2['new_unit_price']
    reasonable = (pattern_2['ratio'] >= 0.5) & (pattern_2['ratio'] <= 1.5)
    print(f"   -> {reasonable.sum()} are reasonable ({reasonable.sum()/len(pattern_2)*100:.1f}%)")
    print("\n   Sample:")
    print(pattern_2[['promo_price', 'new_unit_price', 'fixed', 'ratio']].head(5))

# Pattern 3: XXX.XXX.XXX (e.g., 115.900.092)
pattern_3 = problematic_promo[problematic_promo['promo_price'].str.match(r'^\d{3}\.\d{3}\.\d{3}$', na=False)].copy()
print(f"\n   PATTERN: XXX.XXX.XXX (like 115.900.092)")
print(f"   -> Found {len(pattern_3)} rows")
if len(pattern_3) > 0:
    pattern_3['fixed'] = pd.to_numeric(pattern_3['promo_price'].str.replace(r'\.', '', n=1, regex=True))
    pattern_3['ratio'] = pattern_3['fixed'] / pattern_3['new_unit_price']
    reasonable = (pattern_3['ratio'] >= 0.5) & (pattern_3['ratio'] <= 1.5)
    print(f"   -> {reasonable.sum()} are reasonable ({reasonable.sum()/len(pattern_3)*100:.1f}%)")
    print("\n   Sample:")
    print(pattern_3[['promo_price', 'new_unit_price', 'fixed', 'ratio']].head(5))

# Step 5: Summary
print("\n" + "="*80)
print("SUMMARY & RECOMMENDATION")
print("="*80)
print("\nBased on this analysis, the best strategy appears to be:")
print("1. Remove the FIRST dot from promo_price values with multiple dots")
print("2. This gives reasonable ratios for the majority of cases")
print("\nHowever, this won't fix 100% of cases. You'll need to:")
print("- Create a 'cleaned_promo_price' column")
print("- Flag rows that still look suspicious after cleaning")
print("- Decide how to handle the unfixable rows in your analysis")

# Recommended Fix
print("\n" + "="*80)
print("RECOMMENDED FIX")
print("="*80)
print("\nmerged_df['cleaned_promo_price'] = merged_df['promo_price'].copy()")
print("mask = merged_df['promo_price'].str.count(r'\\.') > 1")
print("merged_df.loc[mask, 'cleaned_promo_price'] = merged_df.loc[mask, 'promo_price'].str.replace(r'\\.', '', n=1, regex=True)")
print("merged_df['cleaned_promo_price'] = pd.to_numeric(merged_df['cleaned_promo_price'], errors='coerce')")

## Documentation

### Dataset Overview
- **19,326 rows** and **7 columns:** `sku`, `name`, `desc`, `price`, `promo_price`, `in_stock`, `type`
- 9,504 duplicates for sku / 9,777 duplicates for name / 14,373 duplicates for desc / 19,318 duplicates for type

### Missing Values
- sku: 0
- name: 0
- desc: 7
- price: 46
- promo_price: 0
- in_stock: 0
- type: 50

### Data Quality Notes
- The prices in `promo_price` appear to be incorrect - for 3-digit prices, the decimal point seems to have shifted one position to the right

### Summary Statistics (without data cleaning)
- **Average price:** $1,862.57 USD
- **Median:** $1,676.94 USD  
- **75th percentile:** $3,349 (three quarters of all products are below this value)